# 06 · E5: comparación confirmatoria N = 100 vs N = 300 (D26, D27; H9 y H10)
Configuración cerrada: D23 (lr 2e-5 / 2e-6) + D25 (lr 1e-4 en la cabeza) + D26 (precisión mixta). No se ajusta nada más.
- Tres semillas por N (42, 43, 44); la semilla fija el orden de los datos y el ruido inicial de las queries. N = 100 y N = 300 con la misma semilla forman un par.
- Validación solo en la época 20. Test-dev (1610 imágenes) se evalúa una única vez, al final de cada corrida, junto con el zero-shot.
- Análisis de H9 con bootstrap pareado sobre las imágenes de test-dev (10 000 remuestreos, semilla 0).

**Antes de ejecutar:** D26, D27, H9 y H10 deben estar commiteadas en `DESIGN_LOG.md`.
Opciones de sesión: Internet activado, **GPU T4 x2**, dataset `banuprasadb/visdrone-dataset`. Ejecutar con *Save Version → Save & Run All*.

In [ ]:
import os, sys, subprocess, time, json
import warnings; warnings.filterwarnings('ignore')
REPO_URL = 'https://github.com/OscarLauLu/DETR-1INF52.git'
BRANCH   = 'rodrigo_dev'
REPO_DIR = 'DETR-1INF52'
if not os.path.exists('../src/train_e3.py'):
    if not os.path.exists(f'{REPO_DIR}/src'):
        !git clone -q -b {BRANCH} {REPO_URL} {REPO_DIR}
    os.chdir(f'{REPO_DIR}/notebooks')
    !pip -q install -r ../requirements.txt
sys.path.insert(0, '../src')
from common import *
import torch, numpy as np, pandas as pd, matplotlib.pyplot as plt
SEED = 42; set_seed(SEED)
print('GPUs:', torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
print(env_report('../results/entorno_e5.txt'))

In [ ]:
# Evidencia: último commit del design log y del script (deben ser ANTERIORES a las corridas de abajo)
!git -C .. log -1 --format='%h  %ci  %s' -- DESIGN_LOG.md
!git -C .. log -1 --format='%h  %ci  %s' -- src/train_e3.py

## C0 · Datos: train (D21), validación y test-dev (D27)

In [ ]:
import glob
cands = sorted(glob.glob('/kaggle/input/**/VisDrone2019-DET-train', recursive=True))
assert cands, 'Dataset no montado: Add Input -> banuprasadb/visdrone-dataset'
ROOT = os.path.dirname(cands[0]); print('ROOT =', ROOT)
TRAIN_IMG, TRAIN_LBL = f'{ROOT}/VisDrone2019-DET-train/images', f'{ROOT}/VisDrone2019-DET-train/labels'
VAL_IMG = f'{ROOT}/VisDrone2019-DET-val/images'
TEST_IMG, TEST_LBL = f'{ROOT}/VisDrone2019-DET-test-dev/images', f'{ROOT}/VisDrone2019-DET-test-dev/labels'
N_TRAIN = 1500
TRAIN_FULL, TRAIN_JSON, VAL_JSON = '/tmp/train_full.json', '../data/train_e3.json', '../data/val_sub.json'
TEST_JSON = '../data/test_dev.json'
OUT, CACHE = '../results/e5', '/tmp/e3cache'
os.makedirs(OUT, exist_ok=True)
t = time.time()
if not os.path.exists(TRAIN_JSON):
    print('conversión de train completo:', yolo_to_coco(TRAIN_IMG, TRAIN_LBL, TRAIN_FULL), f'{time.time() - t:.0f} s')
    print('subconjunto:', make_subset(TRAIN_FULL, N_TRAIN, TRAIN_JSON, seed=SEED))
if not os.path.exists(TEST_JSON):
    t = time.time(); print('conversión de test-dev:', yolo_to_coco(TEST_IMG, TEST_LBL, TEST_JSON), f'{time.time() - t:.0f} s')
for name, js in (('train_e3', TRAIN_JSON), ('val', VAL_JSON), ('test-dev', TEST_JSON)):
    c = np.array(list(objects_per_image(js).values()))
    print(f'{name}: {len(c)} imágenes, {c.sum()} cajas, {c.mean():.1f} por imagen, '
          f'{(c > 100).sum()} con n > 100 ({(c > 100).mean():.1%}), máximo {c.max()}')
# La caché es de 512 x 853 (horizontal): una imagen vertical no cabe (ver prep en train_e3.py)
vert = [im['file_name'] for im in json.load(open(TEST_JSON))['images'] if im['height'] > im['width']]
print('imágenes verticales en test-dev:', len(vert), vert[:5])
assert not vert, 'Hay imágenes verticales en test-dev: no lanzar sin decidir cómo tratarlas'
_, (q1, q2) = density_groups(TEST_JSON)
print(f'test-dev: terciles de objetos por imagen q1 = {q1:.0f}, q2 = {q2:.0f}')

In [ ]:
# D22 · Preprocesamiento único, ahora también de test-dev (D27); los splits ya cacheados se omiten
t = time.time()
if not os.path.exists(f'{CACHE}/test_labels.pt'):
    !python ../src/train_e3.py prep --train_img {TRAIN_IMG} --train_json {TRAIN_JSON} --val_img {VAL_IMG} --val_json {VAL_JSON} --test_img {TEST_IMG} --test_json {TEST_JSON} --cache {CACHE}
print(f'{time.time() - t:.0f} s')
!du -sh {CACHE}
!free -g | head -2

## C1 · Prueba corta (5 iteraciones, 8 imágenes) antes de gastar ~4 h de GPU

In [ ]:
SMOKE = '/tmp/smoke'
r = subprocess.run(['python', '-u', '../src/train_e3.py', 'train', '--N', '300', '--seed', '43', '--cache', CACHE,
                    '--val_json', VAL_JSON, '--test_json', TEST_JSON, '--out', SMOKE, '--epochs', '1',
                    '--max_iters', '5', '--max_val', '8', '--eval_at', '', '--amp', '--lr_head', '1e-4',
                    '--zero_shot', '--save_model'],
                   capture_output=True, text=True, env={**os.environ, 'CUDA_VISIBLE_DEVICES': '0'})
print(r.stdout[-1500:]); print(r.stderr[-3000:])
assert r.returncode == 0, 'La prueba corta falló: no lanzar las corridas'
L0 = json.load(open(f'{SMOKE}/N300.json'))
for f in ('zero_shot.json', 'zero_shot_test.json', 'model_N300.pt', 'dets_N300.json.gz', 'dets_test_N300.json.gz'):
    assert os.path.exists(f'{SMOKE}/{f}'), f
assert 'test' in L0 and '1' in L0['eval'], list(L0)
print('Prueba corta correcta')

## C2 · Corridas: 3 semillas × N ∈ {100, 300}, una por GPU

In [ ]:
SEEDS, NS = [42, 43, 44], [100, 300]
COMMON = ['--lr', '2e-5', '--lr_backbone', '2e-6', '--lr_head', '1e-4', '--lr_drop', '14', '--eval_at', '',
          '--amp', '--test_json', TEST_JSON, '--save_model']
VARIANTS = {f'N{N}_s{s}': ['--N', str(N), '--seed', str(s)] + COMMON + (['--zero_shot'] if (N, s) == (100, 42) else [])
            for s in SEEDS for N in NS}
EPOCHS = 20

def cmd(name):
    return ['python', '-u', '../src/train_e3.py', 'train', '--cache', CACHE, '--val_json', VAL_JSON,
            '--out', f'{OUT}/{name}', '--epochs', str(EPOCHS)] + VARIANTS[name]

n_gpu = torch.cuda.device_count()
procs, gpu_of, t0 = {}, {}, time.time()
def launch(name, gpu):
    os.makedirs(f'{OUT}/{name}', exist_ok=True)
    env = {**os.environ, 'CUDA_VISIBLE_DEVICES': str(gpu)}
    procs[name] = subprocess.Popen(cmd(name), env=env, stdout=open(f'{OUT}/log_{name}.txt', 'w'),
                                   stderr=subprocess.STDOUT)
    gpu_of[name] = gpu

pending = list(VARIANTS)
while pending or any(p.poll() is None for p in procs.values()):
    busy = {gpu_of[n] for n in procs if procs[n].poll() is None}
    for gpu in range(max(1, n_gpu)):
        if pending and gpu not in busy:
            launch(pending.pop(0), gpu); busy.add(gpu)
    time.sleep(120)
    for name in procs:
        if procs[name].poll() is None:
            lines = [l for l in open(f'{OUT}/log_{name}.txt').read().splitlines() if l.startswith('[N')]
            print(f'{(time.time() - t0) / 60:5.1f} min | {name:8s} |', lines[-1] if lines else 'iniciando')

for name, p in procs.items():
    print(f'{name}: código de salida {p.returncode}')
    if p.returncode: print(open(f'{OUT}/log_{name}.txt').read()[-3000:])

## C3 · Resultados en test-dev (H10) y en validación

In [ ]:
L = {n: json.load(open(f'{OUT}/{n}/N{n.split("_")[0][1:]}.json')) for n in VARIANTS}
ZST = json.load(open(f'{OUT}/N100_s42/zero_shot_test.json'))
ZSV = json.load(open(f'{OUT}/N100_s42/zero_shot.json'))
fila = lambda e: {'AP 6cl': e['mapped6']['AP'], 'AP 10cl': e['overall']['AP'], 'AP50 10cl': e['overall']['AP50'],
                  'AR@500': e['overall']['AR@500'], 'AR@500 low': e['low']['AR@500'],
                  'AR@500 gt100': e['gt100']['AR@500']}
T = pd.DataFrame({n: fila(l['test']) for n, l in L.items()}).T
T['N'] = [int(n.split('_')[0][1:]) for n in T.index]
print('test-dev, por corrida'); print(T.drop(columns='N').round(4).to_string())
print('\ntest-dev, media ± desviación estándar entre semillas')
print(T.groupby('N').agg(['mean', 'std']).round(4).to_string())
print(f"\nzero-shot en test-dev (6 clases): AP {ZST['overall']['AP']:.4f} | AP50 {ZST['overall']['AP50']:.4f}")
print(f"zero-shot en validación (6 clases): AP {ZSV['overall']['AP']:.4f}")

ap100 = T.loc[T.N == 100, 'AP 6cl']
print('\nH10: AP en las 6 clases de las tres semillas con N = 100, frente al zero-shot en test-dev')
print('  diferencias:', (ap100 - ZST['overall']['AP']).round(4).to_dict())
print('  H10 se cumple' if (ap100 > ZST['overall']['AP']).all() else '  H10 no se cumple')

print('\nAP por clase en test-dev (10 clases)')
print(pd.DataFrame({n: {c: v['AP'] for c, v in l['test']['por_clase'].items()} for n, l in L.items()}).round(4).to_string())
V = pd.DataFrame({n: fila(l['eval']['20']) for n, l in L.items()}).T
print('\nvalidación, época 20 (continuidad con el E4)'); print(V.round(4).to_string())
T.to_csv(f'{OUT}/tabla_test.csv'); V.to_csv(f'{OUT}/tabla_val.csv')

## C4 · H9: bootstrap pareado sobre las imágenes de test-dev

In [ ]:
G, (q1, q2) = density_groups(TEST_JSON)
GR = {g: np.array(G[g]) for g in ('low', 'gt100')}
print({g: len(v) for g, v in GR.items()}, f'(low: n <= {q1:.0f})')
rng = np.random.default_rng(0); B = 10_000
DRAWS = {g: rng.integers(0, len(v), size=(B, len(v))) for g, v in GR.items()}   # mismos remuestreos para todas las semillas

def recall_boot(run, g):
    """Recall agregado (sum tp / sum n_gt, IoU 0,5) del grupo g: valor observado y B remuestreos."""
    R = {r['image_id']: (r['tp'], r['n_gt']) for r in L[run]['test']['recall_img']}
    tp = np.array([R.get(i, (0, 0))[0] for i in GR[g]], float); n = np.array([R.get(i, (0, 0))[1] for i in GR[g]], float)
    return tp.sum() / n.sum(), tp[DRAWS[g]].sum(1) / n[DRAWS[g]].sum(1)

rows, Dboot = [], []
for s in SEEDS:
    d_obs, d_b = {}, {}
    for g in GR:
        (o100, b100), (o300, b300) = recall_boot(f'N100_s{s}', g), recall_boot(f'N300_s{s}', g)
        d_obs[g], d_b[g] = o300 - o100, b300 - b100
    D, Db = d_obs['gt100'] - d_obs['low'], d_b['gt100'] - d_b['low']
    Dboot.append(Db)
    lo, hi = np.percentile(Db, [2.5, 97.5])
    rows.append({'semilla': s, 'ΔR low': d_obs['low'], 'ΔR gt100': d_obs['gt100'], 'D = ΔR gt100 − ΔR low': D,
                 'IC95 inf': lo, 'IC95 sup': hi})
H9 = pd.DataFrame(rows).set_index('semilla')
Dm = np.mean(Dboot, axis=0); lo, hi = np.percentile(Dm, [2.5, 97.5])
H9.loc['media'] = [H9['ΔR low'].mean(), H9['ΔR gt100'].mean(), H9['D = ΔR gt100 − ΔR low'].mean(), lo, hi]
print(H9.round(4).to_string()); H9.to_csv(f'{OUT}/h9_bootstrap.csv')

if (H9.loc[SEEDS, 'IC95 inf'] > 0).all():
    print('\nH9 (regla registrada): más queries mejoran el recall sobre todo en escenas densas (IC > 0 en las 3 semillas)')
elif hi < 0.01:
    print('\nH9 (regla registrada): más queries NO mejoran más en escenas densas (IC conjunto por debajo de +0,01)')
else:
    print('\nH9 (regla registrada): no concluyente')